# Pitcher name match vs stats table

This notebook mirrors how **`build_dataset`** joins schedule probables to the season pitcher table:

1. **`fetch_pitcher_stats(season)`** — same source as the pipeline (FanGraphs via pybaseball when `USE_FANGRAPHS=1`, else MLB Stats API).
2. **`match_player_names(schedule_name, pitchers_df)`** — exact normalized name, then fuzzy match to the `Name` column.

Use it to see **which probables resolve to a stats row**, which get **no match**, and whether **kbb / xFIP** are present after the join.

**Working directory:** run Jupyter with `mlb-model` as the cwd, or `mlb-model/notebooks` (the first code cell fixes `sys.path`).

**Network:** fetching schedule and stats calls the MLB API and possibly FanGraphs/pybaseball.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd

# Resolve project root (mlb-model/) so `import app` works from notebooks/.
_root = Path.cwd().resolve()
if not (_root / "app").is_dir():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

os.chdir(_root)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 40)

print("Project root:", _root)

Project root: C:\Users\Austin\baseball\mlb-model


## Parameters

**`SEASON`** comes from `PIPELINE_SEASON` or the calendar year. **`START_DATE`** / **`END_DATE`** default to **today → today+7** (inclusive `YYYY-MM-DD`). Override those two variables in the next code cell (after `today = ...`) for a fixed window, then re-run that cell and everything below.

In [2]:
from datetime import datetime

from app.config import FUZZY_MATCH_THRESHOLD, USE_FANGRAPHS
from app.data.fetch_games import fetch_games
from app.data.fetch_stats import fetch_pitcher_stats
from app.utils.name_matching import match_player_names

_ps = os.environ.get("PIPELINE_SEASON", "").strip()
SEASON = int(_ps) if _ps.isdigit() else datetime.now().year

today = pd.Timestamp.today().normalize()
START_DATE = today.strftime("%Y-%m-%d")
END_DATE = (today + pd.Timedelta(days=7)).strftime("%Y-%m-%d")

print(f"SEASON={SEASON}  schedule {START_DATE} .. {END_DATE}")
print(f"USE_FANGRAPHS={USE_FANGRAPHS}  FUZZY_MATCH_THRESHOLD={FUZZY_MATCH_THRESHOLD}")

pitchers = fetch_pitcher_stats(SEASON)
games = fetch_games(START_DATE, END_DATE)

print(f"Pitcher stat rows: {len(pitchers)}  schedule rows: {len(games)}")
if pitchers.empty:
    print("Warning: empty pitcher table — matching will all fail.")
else:
    print("Pitcher columns:", list(pitchers.columns))

SEASON=2026  schedule 2026-04-14 .. 2026-04-21
USE_FANGRAPHS=False  FUZZY_MATCH_THRESHOLD=90
Pitcher stat rows: 467  schedule rows: 110
Pitcher columns: ['id', 'Name', 'Team', 'K%', 'BB%', 'xFIP', 'kbb', 'gamesStarted', 'gamesPlayed', 'is_reliever_season']


## Probable pitcher × stats match

One row per **(game, side)**. **`matched_name`** is the canonical `Name` from the stats table when `match_player_names` succeeds. **`kbb`** / **`xFIP`** are read from that row (same as `build_dataset`); NaN if no row or missing cells.

In [3]:
pitch = pitchers.drop_duplicates(subset=["Name"], keep="first").set_index("Name")

rows: list[dict] = []
if games.empty:
    sp = pd.DataFrame()
else:
    g = games.copy()
    g["game_date"] = pd.to_datetime(g["game_date"], errors="coerce").dt.normalize()
    for _, r in g.iterrows():
        gpk = r.get("game_pk")
        gdate = r.get("game_date")
        for side, team_col, name_col, pid_col in (
            ("home", "home_team_name", "home_probable_pitcher", "home_probable_pitcher_id"),
            ("away", "away_team_name", "away_probable_pitcher", "away_probable_pitcher_id"),
        ):
            sched = r.get(name_col)
            matched = match_player_names(sched, pitchers) if sched else None
            kbb = xfip = float("nan")
            if matched and matched in pitch.index:
                kbb = pitch.loc[matched, "kbb"]
                xfip = pitch.loc[matched, "xFIP"]
            rows.append(
                {
                    "game_pk": gpk,
                    "game_date": gdate,
                    "side": side,
                    "team_name": r.get(team_col),
                    "schedule_pitcher": sched,
                    "mlb_pitcher_id": r.get(pid_col),
                    "matched_name": matched,
                    "match_ok": matched is not None,
                    "kbb": float(kbb) if pd.notna(kbb) else float("nan"),
                    "xFIP": float(xfip) if pd.notna(xfip) else float("nan"),
                    "stats_complete": matched is not None
                    and pd.notna(kbb)
                    and pd.notna(xfip),
                }
            )
    sp = pd.DataFrame(rows)

print("Side-rows:", len(sp))
if not sp.empty:
    display(sp.sort_values(["game_date", "game_pk", "side"]).reset_index(drop=True))

Side-rows: 220


,game_pk,game_date,side,team_name,schedule_pitcher,mlb_pitcher_id,matched_name,match_ok,kbb,xFIP,stats_complete
0,823072,2026-04-14,away,Cleveland Guardians,Joey Cantillo,676282,Joey Cantillo,True,21.666667,1.804545,True
1,823072,2026-04-14,home,St. Louis Cardinals,Michael McGreevy,700241,Michael McGreevy,True,12.903226,3.040000,True
2,823315,2026-04-14,away,Seattle Mariners,Bryan Woo,693433,Bryan Woo,True,19.117647,2.044444,True
3,823315,2026-04-14,home,San Diego Padres,Michael King,650633,Michael King,True,10.294118,3.700000,True
4,823398,2026-04-14,away,Washington Nationals,PJ Poulin,676571,PJ Poulin,True,-2.564103,6.988889,True
...,...,...,...,...,...,...,...,...,...,...,...
215,824690,2026-04-21,home,Chicago Cubs,NaN,<NA>,NaN,False,NaN,NaN,False
216,824773,2026-04-21,away,New York Yankees,NaN,<NA>,NaN,False,NaN,NaN,False
217,824773,2026-04-21,home,Boston Red Sox,NaN,<NA>,NaN,False,NaN,NaN,False
218,825099,2026-04-21,away,Chicago White Sox,NaN,<NA>,NaN,False,NaN,NaN,False


## Summary and failures

- **`match_ok`** — fuzzy/exact match to a `Name` in the stats table.
- **`stats_complete`** — match succeeded **and** `kbb` / `xFIP` are non-null on that row (same idea as pipeline `sp_season_stats_complete` per side, shown here per SP slot).

In [4]:
if sp.empty:
    print("No schedule rows in range — widen dates or check the API.")
else:
    tot = len(sp)
    print(
        "match_ok:",
        sp["match_ok"].sum(),
        "/",
        tot,
        "  stats_complete:",
        sp["stats_complete"].sum(),
        "/",
        tot,
    )

    bad = sp[~sp["stats_complete"]].sort_values(["game_date", "game_pk", "side"])
    print("\nRows needing attention (no match or missing kbb/xFIP):", len(bad))
    if not bad.empty:
        display(
            bad[
                [
                    "game_date",
                    "game_pk",
                    "side",
                    "team_name",
                    "schedule_pitcher",
                    "matched_name",
                    "kbb",
                    "xFIP",
                ]
            ]
        )

match_ok: 75 / 220   stats_complete: 75 / 220

Rows needing attention (no match or missing kbb/xFIP): 145


,game_date,game_pk,side,team_name,schedule_pitcher,matched_name,kbb,xFIP
22,2026-04-14,824210,home,Houston Astros,Colton Gordon,NaN,NaN,NaN
16,2026-04-14,824616,home,Chicago White Sox,Noah Schultz,NaN,NaN,NaN
1,2026-04-14,824853,away,Arizona Diamondbacks,Merrill Kelly,NaN,NaN,NaN
54,2026-04-15,823314,home,San Diego Padres,NaN,NaN,NaN,NaN
38,2026-04-15,823399,home,Pittsburgh Pirates,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
206,2026-04-21,824690,home,Chicago Cubs,NaN,NaN,NaN,NaN
201,2026-04-21,824773,away,New York Yankees,NaN,NaN,NaN,NaN
200,2026-04-21,824773,home,Boston Red Sox,NaN,NaN,NaN,NaN
217,2026-04-21,825099,away,Chicago White Sox,NaN,NaN,NaN,NaN


## Search the stats table by substring

Useful when **`match_ok`** is false: see whether the player exists under a different spelling or is absent from the table entirely.

In [8]:
def find_pitchers(substr: str, df: pd.DataFrame | None = None) -> pd.DataFrame:
    """Case-insensitive substring match on the stats `Name` column."""
    df = df if df is not None else pitchers
    if df.empty or "Name" not in df.columns:
        return pd.DataFrame()
    m = df["Name"].astype(str).str.contains(substr, case=False, na=False)
    return df.loc[m].sort_values("Name")


# Example: replace with a last name from a failed row above
find_pitchers("Shane McClanahan")

,id,Name,Team,K%,BB%,xFIP,kbb,gamesStarted,gamesPlayed,is_reliever_season
251,663556,Shane McClanahan,TBR,25.0,19.444444,3.446154,5.555556,2,2,False


## Optional: compare to saved `games_{season}.parquet`

If the pipeline has written Parquet, compare **`sp_season_stats_complete`** (game-level, both SP slots have finite kbb/xFIP) to this notebook’s side-level view.

In [6]:
from app.config import games_parquet_path

pq = games_parquet_path(SEASON)
if pq.is_file():
    gdf = pd.read_parquet(pq)
    gdf["game_date"] = pd.to_datetime(gdf["game_date"], errors="coerce").dt.normalize()
    in_win = gdf[gdf["game_date"].between(pd.to_datetime(START_DATE), pd.to_datetime(END_DATE))]
    print("Parquet:", pq.resolve(), "rows in date window:", len(in_win))
    if "sp_season_stats_complete" in in_win.columns:
        inc = in_win[~in_win["sp_season_stats_complete"]]
        print("Games with sp_season_stats_complete == False:", len(inc))
        if not inc.empty:
            cols = [
                c
                for c in (
                    "game_date",
                    "game_pk",
                    "home_team_name",
                    "away_team_name",
                    "home_probable_pitcher",
                    "away_probable_pitcher",
                    "home_sp_kbb",
                    "away_sp_kbb",
                    "home_sp_xfip",
                    "away_sp_xfip",
                    "sp_season_stats_complete",
                )
                if c in inc.columns
            ]
            display(inc[cols].sort_values("game_date"))
    else:
        print("Column sp_season_stats_complete missing — rebuild dataset with a newer pipeline.")
else:
    print("No parquet at", pq, "(run live/backfill to create it).")

Parquet: C:\Users\Austin\baseball\mlb-model\data\games_2026.parquet rows in date window: 110
Games with sp_season_stats_complete == False: 82


,game_date,game_pk,home_team_name,away_team_name,home_probable_pitcher,away_probable_pitcher,home_sp_kbb,away_sp_kbb,home_sp_xfip,away_sp_xfip,sp_season_stats_complete
609,2026-04-14,824853,Baltimore Orioles,Arizona Diamondbacks,Trevor Rogers,Merrill Kelly,11.842105,NaN,2.573684,NaN,False
617,2026-04-14,824616,Chicago White Sox,Tampa Bay Rays,Noah Schultz,Shane McClanahan,NaN,5.555556,NaN,3.446154,False
620,2026-04-14,824210,Houston Astros,Colorado Rockies,Colton Gordon,Michael Lorenzen,NaN,8.000000,NaN,5.957143,False
628,2026-04-15,823399,Pittsburgh Pirates,Washington Nationals,NaN,Jake Irvin,NaN,12.500000,NaN,5.314286,False
631,2026-04-15,823561,New York Yankees,Los Angeles Angels,Luis Gil,NaN,-5.263158,NaN,8.350000,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...
705,2026-04-21,822995,Tampa Bay Rays,Cincinnati Reds,NaN,NaN,NaN,NaN,NaN,NaN,False
704,2026-04-21,824448,Cleveland Guardians,Houston Astros,NaN,NaN,NaN,NaN,NaN,NaN,False
717,2026-04-21,825099,Arizona Diamondbacks,Chicago White Sox,NaN,NaN,NaN,NaN,NaN,NaN,False
710,2026-04-21,823640,New York Mets,Minnesota Twins,NaN,NaN,NaN,NaN,NaN,NaN,False


## Game-level rollup

**`both_sp_stats_complete`** — both home and away side-rows have `stats_complete` (same spirit as Parquet `sp_season_stats_complete` after a successful pipeline rebuild).

In [ ]:
if sp.empty:
    print("(no side-rows)")
else:
    piv = sp.pivot_table(
        index=["game_date", "game_pk"],
        columns="side",
        values=["team_name", "schedule_pitcher", "matched_name", "stats_complete"],
        aggfunc="first",
    )
    piv.columns = [f"{a}_{b}" for a, b in piv.columns]
    piv = piv.reset_index()
    sch, sca = "stats_complete_home", "stats_complete_away"
    if sch in piv.columns and sca in piv.columns:
        piv["both_sp_stats_complete"] = piv[sch].fillna(False) & piv[sca].fillna(False)
    else:
        piv["both_sp_stats_complete"] = False
    print("Games in window:", len(piv))
    print("both_sp_stats_complete:", int(piv["both_sp_stats_complete"].sum()))
    weak = piv[~piv["both_sp_stats_complete"]].sort_values("game_date")
    if not weak.empty:
        print("\nGames where at least one SP slot lacks stats:")
        show_cols = [
            c
            for c in piv.columns
            if c
            in ("game_date", "game_pk", "both_sp_stats_complete", "team_name_home", "team_name_away")
            or "schedule_pitcher" in c
            or "matched_name" in c
            or c in (sch, sca)
        ]
        display(weak[show_cols])